# I2. Interferometry and visibilities

The second notebook of the introductory track ({doc}`../index`). It uses the
public SKA pointing fixture: 104 s of four MeerKAT core antennas on
PKS 1934−638, 4,096 channels (45 MB, downloaded once and checked against its
SHA-256). It runs in about half a minute.

**Learning goals.** After this notebook you can

- write the geometric delay of a two-element interferometer and the fringe it
  produces;
- say what a visibility is: one Fourier component of the sky brightness (the
  van Cittert-Zernike theorem), sampled at a baseline measured in wavelengths
  (u, v, w);
- predict the amplitude and phase of a point source at and away from the
  phase centre, and the amplitude of a resolved source;
- read the amplitude and phase of real MeerKAT visibilities, and estimate the
  system equivalent flux density (I1) from autocorrelations against
  cross-correlations.

In [ ]:
%matplotlib inline
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from rang import simtruth as st
from rang.notebook import data, flags

C = 299_792_458.0
ARCMIN = math.pi / 180 / 60
warnings.filterwarnings("ignore", "Mean of empty slice")

# The fixture: RANG_FIXTURE or a cached copy, else downloaded once and SHA-256 checked.
ds = data.load_fixture(data.find_fixture() or data.download_fixture(data.default_cache_dir()))
s = ds.state
pd.Series(data.summary(ds))

## 1. Two antennas and a geometric delay

Two antennas separated by a baseline vector **b** look at a source in
direction **ŝ**. The wavefront reaches one antenna later than the other by the
**geometric delay** τ_g = **b**·**ŝ** / c. Multiplying and averaging the two
voltages (correlating them) gives a response ∝ cos(2π ν τ_g): a **fringe**.
As the Earth turns, τ_g changes and the fringe oscillates.

In the coordinates radio astronomers use, (u, v, w) is the baseline vector in
metres with w pointing at the phase centre, so τ_g = w / c. The fixture's UVW
column gives it for every row:

In [ ]:
uvw = ds.extra["uvw"]
a1, a2 = s["antenna1"], s["antenna2"]
t = s["times"] - s["times"].min()
cross = a1 != a2
nu0 = 1.284e9
rows = []
for p, q in ds.baselines():
    sel = (a1 == p) & (a2 == q)
    w = uvw[sel, 2]
    rate = np.polyfit(t[sel], w, 1)[0]  # m/s
    rows.append(
        {
            "baseline": ds.baseline_label((p, q)),
            "length (m)": round(float(np.linalg.norm(uvw[sel][0])), 1),
            "tau_g at the start (ns)": round(float(w[0] / C * 1e9), 2),
            "fringe rate at 1.284 GHz (mHz)": round(float(nu0 * rate / C * 1e3), 2),
        }
    )
pd.DataFrame(rows)

The correlator removes this delay and its rate for the **phase centre**
(delay tracking and fringe stopping), so a source exactly at the phase centre
gives a visibility of constant phase. A source elsewhere keeps a residual
fringe. What is left on real data is the instrument: section 4.

## 2. A visibility is a Fourier component of the sky

For a sky brightness I(l, m), with (l, m) the direction cosines relative to
the phase centre and n = √(1 − l² − m²), the **visibility** on a baseline (u, v,
w) in wavelengths is

V(u, v, w) = ∫∫ I(l, m) / n · exp(2πi [u l + v m + w (n − 1)]) dl dm.

(rang and the A1 simulation use this positive sign; some texts use the
negative one.) For a small field, w (n − 1) is negligible and V is the 2D
Fourier transform of the sky: the **van Cittert-Zernike theorem**. Each
baseline at each frequency measures one Fourier component.

rang predicts visibilities of point and Gaussian components with its Rust
DFT (`rang.simtruth.target_coherency`). For a 1 Jy point source at the phase
centre and one 2′ away, on the fixture's baselines:

In [ ]:
def component(l=0.0, m=0.0, flux=1.0, fwhm_arcsec=0.0):
    return {
        "kind": "gaussian" if fwhm_arcsec else "point",
        "l": l, "m": m, "flux": flux, "alpha": 0.0, "fwhm_arcsec": fwhm_arcsec,
        "p": 0.0, "chi0_deg": 0.0, "rm": 0.0, "ref_freq_hz": nu0,
    }


freq = s["frequencies"]
l_off = 2 * ARCMIN
centre = st.target_coherency(uvw[cross], freq, [component()])[..., 0]
offset = st.target_coherency(uvw[cross], freq, [component(l=l_off)])[..., 0]
n = math.sqrt(1 - l_off**2)
expected = np.exp(2j * np.pi * (uvw[cross, 0, None] * l_off + uvw[cross, 2, None] * (n - 1)) * freq[None] / C)
print(f"centre: |V| from {np.abs(centre).min():.6f} to {np.abs(centre).max():.6f}, max |phase| {np.abs(np.angle(centre)).max():.2e} rad")
print(f"2 arcmin east: max |V_rang - exp(2 pi i (u l + w (n - 1)))| = {np.abs(offset - expected).max():.2e}")

The offset source's phase winds with u l, so it turns faster with frequency on
longer baselines. That is how an interferometer measures position:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for k, (p, q) in enumerate(ds.baselines()[:3]):
    row = np.flatnonzero((a1[cross] == p) & (a2[cross] == q))[0]
    ax.plot(freq / 1e6, np.degrees(np.angle(offset[row])), label=f"{ds.baseline_label((p, q))}, u = {uvw[cross][row, 0]:.1f} m")
ax.set(xlabel="frequency (MHz)", ylabel="phase (deg)", title="1 Jy point source 2′ east of the phase centre")
ax.legend()
plt.show()

A resolved source loses amplitude on long baselines. For a circular
Gaussian of FWHM θ, the visibility amplitude at baseline length ρ wavelengths
is exp(−(π θ ρ)² / (4 ln 2)). rang's DFT against that formula, for θ = 30″:

In [ ]:
rho = np.linspace(0, 20e3, 200)  # wavelengths
line = np.stack([rho, np.zeros_like(rho), np.zeros_like(rho)], -1) * C / nu0  # metres at nu0
theta = 30 / 3600 * math.pi / 180
gauss = np.abs(st.target_coherency(line, np.array([nu0]), [component(fwhm_arcsec=30.0)])[:, 0, 0])
analytic = np.exp(-((np.pi * theta * rho) ** 2) / (4 * math.log(2)))
print(f"max |DFT - analytic| = {np.abs(gauss - analytic).max():.2e}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(rho / 1e3, gauss, label="rang DFT")
ax.plot(rho / 1e3, analytic, "k--", label="analytic")
ax.set(xlabel="baseline (kλ)", ylabel="|V| (Jy)", title="1 Jy Gaussian, FWHM 30″")
ax.legend()
plt.show()

Short baselines see large structure, long baselines fine structure. The
fixture's longest baseline, in wavelengths at the band edges:

In [ ]:
length = np.linalg.norm(uvw[cross, :2], axis=1).max()
for f in (freq[0], freq[-1]):
    lam = C / f
    print(f"{f / 1e6:.0f} MHz: {length:.1f} m = {length / lam:.0f} wavelengths, fringe spacing λ/B = {lam / length * 180 / math.pi * 60:.1f}′")

PKS 1934−638 is far smaller than these fringe spacings: to these baselines it
is a point source, which is why it calibrates.

## 3. Real visibilities

A waterfall plots one baseline's visibilities against time and frequency.
Grey cells are flagged in the archive, mostly RFI (notebook I5):

In [ ]:
display(flags.waterfall(ds, (0, 1), "amplitude"))  # rang.notebook figures are returned, not shown
display(flags.waterfall(ds, (0, 1), "phase"))

The amplitude follows the receivers' bandpass, not the source's smooth
spectrum, and the phase is not flat: the instrument multiplies every
visibility by complex gains. Notebook I4 writes that down as the
measurement equation.

## 4. What the phase says: instrumental delay

A residual delay τ between two signal paths gives a phase 2π ν τ that grows
linearly with frequency. The mean phase change from one channel to the next
measures it:

In [ ]:
f = s["original_flags"]
dnu = freq[1] - freq[0]
rows = []
for p, q in ds.baselines():
    sel = (a1 == p) & (a2 == q)
    out = {"baseline": ds.baseline_label((p, q))}
    for c, name in ((0, "XX"), (3, "YY")):
        v = np.where(f[sel][..., c], 0, s["data"][sel][..., c])
        step = np.sum(v[:, 1:] * np.conj(v[:, :-1]))
        out[f"{name} delay (ns)"] = round(float(np.angle(step) / (2 * np.pi * dnu) * 1e9), 2)
    rows.append(out)
pd.DataFrame(rows)

Delays of a few nanoseconds: the K term that calibration solves first
(I4, I6).

## 5. Autocorrelations and the SEFD

The fixture also has **autocorrelations**, each antenna correlated with
itself. An autocorrelation measures total power: the receiver's noise plus
the source, g_p² (SEFD_p + S). A cross-correlation keeps only what is
correlated between the antennas, the source: g_p g_q S. The gains cancel in

√(A_pp A_qq) / |V_pq| = √((SEFD_p + S)(SEFD_q + S)) / S,

so the data, together with the flux model of I1, measure the SEFD. Averaged
over time, per channel, then the median over unflagged channels:

In [ ]:
S = data.reynolds_flux(freq)


def mean_amp(sel, c):
    return np.nanmean(np.where(f[sel][..., c], np.nan, np.abs(s["data"][sel][..., c])), axis=0)


print(f"autocorrelation rows: {(a1 == a2).sum()}, cross-correlation rows: {cross.sum()}")
rows = []
for p, q in ds.baselines():
    sel, pp, qq = (a1 == p) & (a2 == q), (a1 == p) & (a2 == p), (a1 == q) & (a2 == q)
    out = {"baseline": ds.baseline_label((p, q))}
    for c, name in ((0, "XX"), (3, "YY")):
        sefd = np.sqrt(mean_amp(pp, c) * mean_amp(qq, c)) / mean_amp(sel, c) * S - S
        out[f"SEFD {name} (Jy)"] = round(float(np.nanmedian(sefd)))
    rows.append(out)
table = pd.DataFrame(rows)
table

In [ ]:
print(f"median over baselines and hands: {np.median(table.iloc[:, 1:].to_numpy()):.0f} Jy; A1 assumes 420 Jy (I1)")

This assumes that the source is the only correlated signal, that it is
unresolved and that the Reynolds (1994) model holds; RFI that survives the
archive's flags biases individual channels, hence the median.

## Exercises

1. Use the fixture's UVW to find the hour angle range of the scan from how
   (u, v) rotate. How many degrees does the baseline turn in 104 s?
2. Predict the visibilities of two equal point sources 5′ apart with
   `target_coherency`. At which baseline length in wavelengths does the
   amplitude first reach zero? Compare with 1/(separation in radians).
3. Repeat the delay table on the cross-hands XY and YX. What do you expect
   for an unpolarised source?
4. Plot the SEFD estimate against frequency for one antenna pair. Where does
   it rise, and why might it (RFI, the bandpass edges, the receiver)?

## Where next

- {doc}`i3-meerkat-array`: the whole array and its uv coverage.
- Notebook {doc}`01-data-and-rank` loads the same data and checks it against
  the archive; {doc}`../../concepts/ingest` covers Measurement Sets.
- {doc}`i1-radio-sky` for flux density and the radiometer equation.